# Create Initial/Final Materials

Build an ordered initial → (optional intermediates) → final set of materials from one starting structure, and write them to a subfolder under `uploads/` in path order for [`utils_create_material_set.ipynb`](utils_create_material_set.ipynb).

Any calculation that takes an ordered start/end pair can use the output; a Nudged Elastic Band path ([`neb.ipynb`](workflows/neb.ipynb)) is one consumer.

Order is preserved by **numbering material names** (`00_...`, `01_...`, …): `utils_create_material_set.ipynb` (and `load_materials_from_folder`) sort by filename, and filenames come from material names.

## Usage

1. Set the material and the names in cell 1.2.
1. Run 2.1, copy the coordinates of the atom you want to move, and paste them into 2.2.
1. Run the rest to build and write the path materials.
1. Open [`utils_create_material_set.ipynb`](utils_create_material_set.ipynb), set the same `SUBFOLDER_NAME` and `IS_ORDERED = True`, and run it to save the materials and create the platform set.
1. Use the printed set name as `MATERIAL_SET` in [`neb.ipynb`](workflows/neb.ipynb).

## Summary

1. Install packages and set parameters.
1. Load the starting material.
1. Clone it as the initial image; move one atom to make the final image.
1. Name members in path order and write them to `uploads/<SUBFOLDER_NAME>/`.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")


### 1.2. Set parameters


In [ ]:
# Starting material (uploads folder or Standata name match)
FOLDER = "uploads"
MATERIAL_NAME = "Silicon (100) surface"

# Short base name for the written images: 00_<PATH_NAME>.json, 01_<PATH_NAME>.json.
PATH_NAME = "initial-final-materials"

# Subfolder under uploads/ to write path materials into — use the same value as
SUBFOLDER_NAME = "initial_final_materials"

## 2. Build path materials
### 2.1. Load the starting material

To read an atom's coordinates from the viewer below: open **Measurements** (the ruler icon), turn on
**Copy Coordinates [C]** — or press `C` — then click the atom. Its coordinates go to the clipboard,
ready to paste into 2.2.

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

source_material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME)
)
visualize(source_material, viewer="wave")


### 2.2. Choose the atom to move

`ATOM_COORDINATE` is the atom you copied above, in crystal coordinates; the nearest atom to it is the
one that moves.

`TRANSLATION` is vector to move it, in Ångström.

In [ ]:
# Coordinates of the atom to move, copied from the viewer above.
ATOM_COORDINATE = [0.0, 0.5, 0.5633]

# Displacement in Angstrom.
TRANSLATION = [0.0, 0.0, -2.0]

### 2.3. Clone as the initial image


In [ ]:
initial_material = source_material.clone()
visualize(initial_material, rotation="-90x")


### 2.4. Transform into the final image (example)

Default: move the atom at `ATOM_COORDINATE` by `TRANSLATION`. Replace with any other transformation (defects, swaps, custom coordinates, …).

2.5 below shows the same move done as a displacement *field* instead, which is what you want when the neighbours should relax along with the atom.


In [ ]:
from mat3ra.made.tools.analyze.other import get_closest_site_id_from_coordinate
from mat3ra.made.tools.operations.core.unary import translate_atoms

atom_id = get_closest_site_id_from_coordinate(initial_material, ATOM_COORDINATE)
final_material = translate_atoms(initial_material, [atom_id], TRANSLATION)

visualize([initial_material, final_material], viewer="wave")

### 2.5. Alternative: move the atom with a perturbation function (example)

`translate_atoms` above applies a fixed vector. A perturbation function instead returns `∆z` for
*every* atom from `f(x, y, z)`, so a Gaussian centred on one atom keeps the displacement local to
it — and lets the path be shaped (neighbours relaxing along with it, a wave, a decaying tail)
rather than a rigid shift.

Left commented out so "Run All" uses the simple translation. Uncomment it to overwrite
`final_material`, run, and compare the two structures in the viewer.

In [ ]:
# import numpy as np
# import sympy as sp
# from mat3ra.made.tools.build_components.operations.core.modifications.perturb import FunctionHolder
# from mat3ra.made.tools.operations.core.unary import perturb
#
# SIGMA = 0.5  # Angstrom — how tightly the displacement is localised around the atom
#
# center_x, center_y, center_z = np.array(ATOM_COORDINATE) @ np.array(
#     initial_material.lattice.vector_arrays
# )
#
# x, y, z = sp.symbols("x y z")
# displacement_function = TRANSLATION[2] * sp.exp(
#     -(((x - center_x) ** 2 + (y - center_y) ** 2 + (z - center_z) ** 2) / (2 * SIGMA**2))
# )
#
# final_material = perturb(
#     initial_material,
#     FunctionHolder(function=displacement_function),
#     use_cartesian_coordinates=True,
# )
# visualize([initial_material, final_material], viewer="wave")

### 2.6. Name members in path order and write to the subfolder

Numeric prefixes control load order in `utils_create_material_set.ipynb` (filenames are sorted; filenames come from material names).


In [ ]:
from mat3ra.notebooks_utils.material import set_materials
from mat3ra.notebooks_utils.settings import UPLOADS_FOLDER

path_materials = [initial_material, final_material]
for index, material in enumerate(path_materials):
    material.name = f"{index:02d}_{PATH_NAME}"

subfolder_path = f"{UPLOADS_FOLDER}/{SUBFOLDER_NAME}"
set_materials(path_materials, folder_path=subfolder_path)
